# RAG pipeline walkthrough
Run the cells top to bottom. Start Jupyter from the **project root folder** and put at least one PDF and one DOCX in `data/corpus/`.
Each step: **what** it is, **why** we need it, then the real code from `app/`.

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
from pathlib import Path
PDF = sorted(Path('data/corpus').glob('*.pdf'))[0]
DOCX = sorted(Path('data/corpus').glob('*.docx'))[0]
PDF, DOCX

(WindowsPath('data/corpus/University.pdf'),
 WindowsPath('data/corpus/University.docx'))

## 1. PDF text extraction
*What:* read text from every page. *Why:* the LLM can only read text. Scanned PDFs return nothing (they need OCR).

In [2]:
from app.ingestion.parsers import parse_pdf, parse_docx
pdf_sections = parse_pdf(PDF.read_bytes())
print(len(pdf_sections), 'pages with text')
print(pdf_sections[0].location); print(pdf_sections[0].text[:500])

2 pages with text
Page 1
Greenfield University 
Student Services Guide 2026 
Document Type: Student Information Guide 
University: Greenfield University 
Academic Year: 2026 
1. University Library 
Greenfield University Library is located in Building A, on the first floor. 
Library opening hours are: 
• Monday to Friday: 8:00 AM to 8:00 PM 
• Saturday: 9:00 AM to 4:00 PM 
• Sunday: Closed 
Students can borrow up to 5 books at a time. The standard borrowing period is 14 
days. 
Books may be renewed once for another 14 da


## 2. DOCX text extraction
DOCX has no fixed pages, so we split by headings and call them *sections*.

In [3]:
docx_sections = parse_docx(DOCX.read_bytes())
print(len(docx_sections), 'sections')
for s in docx_sections[:5]: print(s.location, '|', len(s.text), 'chars')

2 sections
Section 1: Start of document | 2187 chars
Section 2: Tables | 225 chars


## 3. Chunking
*Why:* whole documents are too big to embed or send to an LLM. Small overlapping pieces keep each idea together.

In [4]:
from app.ingestion.chunking import chunk_sections
chunks = chunk_sections(pdf_sections, 'demo', PDF.name, size=800, overlap=120)
print(len(chunks), 'chunks'); print(chunks[0])

5 chunks
Chunk(chunk_id='demo:0', doc_id='demo', filename='University.pdf', page=1, location='Page 1', position=0, text='Greenfield University \nStudent Services Guide 2026 \nDocument Type: Student Information Guide \nUniversity: Greenfield University \nAcademic Year: 2026 \n1. University Library \nGreenfield University Library is located in Building A, on the first floor. Library opening hours are: \n• Monday to Friday: 8:00 AM to 8:00 PM \n• Saturday: 9:00 AM to 4:00 PM \n• Sunday: Closed \nStudents can borrow up to 5 books at a time. The standard borrowing period is 14 \ndays. Books may be renewed once for another 14 days, provided that no other student \nhas reserved the book. The late return fine is Rs. 50 per book per day. Students must show their university ID card to borrow books. 2. Student IT Support \nThe IT Support Office is located in Building B, Room 204.')


## 4. Embedding generation
*What:* each chunk becomes a list of numbers (384 for MiniLM). Similar meaning gives similar numbers.

In [7]:
from app.config import Settings
from app.services.embeddings import build_embedder
settings = Settings(chroma_dir='data/notebook_chroma')
embedder = build_embedder(settings)
vectors = embedder.embed([c.text for c in chunks])
print(len(vectors), 'vectors of size', len(vectors[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

5 vectors of size 384


## 5. Vector storage (ChromaDB)
We store vector + text + metadata (document, page, position).

In [8]:
from app.services.vector_store import VectorStore
store = VectorStore('data/notebook_chroma', collection='notebook_demo')
store.add_chunks(chunks, vectors, file_hash='demo-hash')
store.count(), store.list_documents()

(5,
 [{'doc_id': 'demo',
   'filename': 'University.pdf',
   'file_hash': 'demo-hash',
   'uploaded_at': '2026-11-23T03:50:10+00:00',
   'chunk_count': 5,
   'page_count': 2}])

## 6. Retrieval
Embed the question, find the closest chunks. The score is cosine similarity from ChromaDB.

In [11]:
from app.retrieval import Retriever
retriever = Retriever(embedder, store, settings)
question ='What are the library opening hours at Greenfield University?'
hits = retriever.retrieve(question, top_k=5)
for h in hits: print(round(h.score, 3), h.filename, h.location, '|', h.text[:100].replace('\n', ' '))

0.69 University.pdf Page 1 | Greenfield University  Student Services Guide 2026  Document Type: Student Information Guide  Univer
0.575 University.pdf Page 2 | Scholarship applications submitted after the deadline will not be considered. 4. Student Affairs Off
0.51 University.pdf Page 1 | university ID card to borrow books. 2. Student IT Support  The IT Support Office is located in Build
0.464 University.pdf Page 2 | 101 affairs@greenfield.example  This guide contains general student service information for the 2026
0.444 University.pdf Page 1 | days. 3. Student Scholarships  Greenfield University offers a Merit Scholarship for eligible undergr


## 7. Answer generation
Needs `LLM_API_KEY` in your `.env`. The prompt contains only the selected chunks.

In [12]:
from app.generation import build_messages, generate
from app.services.llm import LLMClient
context = retriever.select_context(hits)
print(build_messages(question, context)[1]['content'][:800])
raw = generate(LLMClient(settings), question, context) if context else 'No chunk passed the score threshold -> refuse'
raw

Documents:
<document id="1" source="University.pdf" location="Page 1">
Greenfield University 
Student Services Guide 2026 
Document Type: Student Information Guide 
University: Greenfield University 
Academic Year: 2026 
1. University Library 
Greenfield University Library is located in Building A, on the first floor. Library opening hours are: 
• Monday to Friday: 8:00 AM to 8:00 PM 
• Saturday: 9:00 AM to 4:00 PM 
• Sunday: Closed 
Students can borrow up to 5 books at a time. The standard borrowing period is 14 
days. Books may be renewed once for another 14 days, provided that no other student 
has reserved the book. The late return fine is Rs. 50 per book per day. Students must show their university ID card to borrow books. 2. Student IT Support 
The IT Support Office is located in Bui


'The Greenfield University Library is open:  \n- **Monday to Friday:** 8:00\u202fAM – 8:00\u202fPM  \n- **Saturday:** 9:00\u202fAM – 4:00\u202fPM  \n- **Sunday:** Closed\u202f【1】'

## 8. Citation handling
The output guardrail removes citation numbers that were never retrieved and refuses answers with no valid citation.

In [13]:
from app.guardrails import validate_answer
checked = validate_answer(raw, [c.text for c in context], settings) if context else None
print(checked)
# Demo of an invented citation:
print(validate_answer('Some claim [1] and a fake one [9]', ['Some claim text'], settings))

ValidatedAnswer(ok=False, text='', cited=[], reason='no_valid_citation')
ValidatedAnswer(ok=True, text='Some claim [1] and a fake one', cited=[1], reason='')
